# MCR Stage D — Density-based reward (GMM per layer)

**Goal**: fit `p_C^ℓ(z)` and `p_W^ℓ(z)` GMMs per layer ℓ ∈ {L11, L17, L23}, operationalize `R_density^ℓ = σ(log p_C − log p_W) − 0.5`.

**Approach**:
1. Load all 1000 Stage B rollouts, pool mean-over-response per layer → `[1000, 2048]` per layer.
2. 80/20 stratified train/val split on outcome.
3. Per layer: PCA(d=64) → GMM(K=3, diag) for correct; GMM(K=3, diag) for wrong.
4. Score on val: AUROC of log-ratio `R_density^ℓ` vs outcome.
5. Multi-layer ensemble: mean of three `R_density^ℓ`.
6. Baseline comparison: L2 logistic regression on same PCA features.

**Decision rules**:
- SHIP R_density if: best-layer AUROC > 0.70 AND multi-layer ensemble AUROC > single-layer by ≥0.02.
- If LogReg beats GMM → density formulation doesn't help, use linear probe instead.
- If AUROC < 0.60 on all layers → pooled residual doesn't separate classes, need token-level features.

**Why raw residuals, not SAE**: SAE at L11/L17 is Stage C (not done yet). For classification we don't need interpretability; PCA(raw) is sufficient. If L23-SAE-PCA beats L23-raw-PCA significantly, we revisit after Stage C.

**Runs on**: CPU. ~15 min (dominated by pooling 1000 rollouts × 3 layers from HF cache).

**Output**: `mcr_d_density_analysis.json` + pickled GMMs + PCAs.

## Cell 1 — Install + imports

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub==1.5.0', 'safetensors', 'scipy',
                'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'], check=True)

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import json, re, pickle, time
from pathlib import Path
import numpy as np
import torch
import pandas as pd
from safetensors import safe_open
from huggingface_hub import hf_hub_download, list_repo_files
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from scipy.special import expit as sigmoid
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

## Cell 2 — CFG

In [ ]:
CFG = dict(
    hf_repo='caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    layers=[11, 17, 23],
    pca_dim=64,
    gmm_k=3,
    gmm_cov='diag',          # diag: 64+64 params/component (vs full: 2145). 366 correct samples → diag safer.
    val_frac=0.2,
    seed=42,
    min_auroc_ship=0.70,
    min_ensemble_gain=0.02,
)
print(json.dumps(CFG, indent=2))

## Cell 3 — Load + pool per-response activations

For each of 1000 rollouts, compute `mean(acts^ℓ[offset_r:offset_r+L_r])` → `[d_model]`. Result per layer: `[1000, 2048]` + outcome labels.

In [ ]:
files = list_repo_files(CFG['hf_repo'], repo_type='dataset')
shards = sorted([f for f in files if re.match(r'shards/q\d{4}\.safetensors$', f)])
print(f'{len(shards)} shards in repo')

pooled = {li: [] for li in CFG['layers']}
outcomes = []
q_idxs = []
disciplines = []

for shard in tqdm(shards, desc='Pool per-response'):
    local = hf_hub_download(CFG['hf_repo'], shard, repo_type='dataset', force_download=False)
    with safe_open(local, framework='pt') as f:
        meta = f.metadata() or {}
        offsets_all = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        q_idx = int(meta['question_global_idx'])
        disc = meta.get('discipline', '')
        for li in CFG['layers']:
            acts = f.get_tensor(f'acts_L{li}')  # [total_tok, d]
            offsets = offsets_all[f'L{li}']
            for r_idx, r in enumerate(rollouts):
                pooled_r = acts[offsets[r_idx]:offsets[r_idx+1]].float().mean(dim=0).numpy()
                pooled[li].append(pooled_r)
        for r in rollouts:
            outcomes.append(int(r['correct']))
            q_idxs.append(q_idx)
            disciplines.append(disc)

# Stack: layer pooled has repeats (one per rollout per layer loop), trim
for li in CFG['layers']:
    pooled[li] = np.stack(pooled[li])
outcomes = np.array(outcomes[:len(pooled[CFG['layers'][0]])])  # de-duplicate
q_idxs = np.array(q_idxs[:len(outcomes)])
disciplines = np.array(disciplines[:len(outcomes)])

# Sanity: lengths should all match n=1000
for li in CFG['layers']:
    assert len(pooled[li]) == 1000, f'L{li}: got {len(pooled[li])} rollouts'
assert len(outcomes) == 1000

print(f'\nPooled shape per layer: {pooled[CFG["layers"][0]].shape}')
print(f'Outcomes: {outcomes.sum()} correct / {len(outcomes) - outcomes.sum()} wrong')

## Cell 4 — Train/val split (80/20, stratified on outcome)

In [ ]:
idx = np.arange(1000)
idx_train, idx_val = train_test_split(idx, test_size=CFG['val_frac'],
                                      stratify=outcomes, random_state=CFG['seed'])
print(f'Train: {len(idx_train)}  ({outcomes[idx_train].sum()} correct, {100*outcomes[idx_train].mean():.1f}%)')
print(f'Val:   {len(idx_val)}   ({outcomes[idx_val].sum()} correct, {100*outcomes[idx_val].mean():.1f}%)')

## Cell 5 — Per-layer pipeline: scale + PCA + GMM(C) + GMM(W)

In [ ]:
def fit_layer_density(X_train, y_train, X_val, y_val, pca_dim, K, cov):
    """Returns dict with scaler, pca, gmm_c, gmm_w, val log-ratios, val AUROC."""
    scaler = StandardScaler().fit(X_train)
    Xt = scaler.transform(X_train)
    Xv = scaler.transform(X_val)
    pca = PCA(n_components=pca_dim, random_state=42).fit(Xt)
    Zt = pca.transform(Xt)
    Zv = pca.transform(Xv)
    # Split by outcome
    Zt_c = Zt[y_train == 1]
    Zt_w = Zt[y_train == 0]
    # Fit GMMs
    gmm_c = GaussianMixture(n_components=K, covariance_type=cov, random_state=42, reg_covar=1e-4).fit(Zt_c)
    gmm_w = GaussianMixture(n_components=K, covariance_type=cov, random_state=42, reg_covar=1e-4).fit(Zt_w)
    # Log-ratio on val
    logp_c = gmm_c.score_samples(Zv)
    logp_w = gmm_w.score_samples(Zv)
    log_ratio = logp_c - logp_w
    # Bounded version: sigmoid
    r_density = sigmoid(log_ratio) - 0.5  # in [-0.5, 0.5]
    auroc = roc_auc_score(y_val, log_ratio)
    # Also fit logistic regression baseline
    logreg = LogisticRegression(C=1.0, max_iter=2000, random_state=42).fit(Zt, y_train)
    logreg_scores = logreg.decision_function(Zv)
    logreg_auroc = roc_auc_score(y_val, logreg_scores)
    # PCA variance captured
    var_kept = pca.explained_variance_ratio_.sum()
    return dict(
        scaler=scaler, pca=pca, gmm_c=gmm_c, gmm_w=gmm_w, logreg=logreg,
        log_ratio=log_ratio, r_density=r_density, logreg_scores=logreg_scores,
        auroc=auroc, logreg_auroc=logreg_auroc, var_kept=var_kept,
    )

results = {}
for li in CFG['layers']:
    print(f'\nLayer L{li}:')
    X_train = pooled[li][idx_train]
    X_val = pooled[li][idx_val]
    y_train = outcomes[idx_train]
    y_val = outcomes[idx_val]
    r = fit_layer_density(X_train, y_train, X_val, y_val,
                          CFG['pca_dim'], CFG['gmm_k'], CFG['gmm_cov'])
    results[li] = r
    print(f'  PCA var kept: {r["var_kept"]:.3f}')
    print(f'  GMM AUROC:    {r["auroc"]:.4f}')
    print(f'  LogReg AUROC: {r["logreg_auroc"]:.4f}')
    print(f'  Δ GMM - LogReg: {r["auroc"] - r["logreg_auroc"]:+.4f}')


## Cell 6 — Multi-layer ensemble

Mean of per-layer log-ratios. Tests the direction-3 hypothesis: do layers add independent signal?

In [ ]:
# Concat log-ratios across layers, uniform-mean ensemble
y_val = outcomes[idx_val]
ensemble_logratio = np.stack([results[li]['log_ratio'] for li in CFG['layers']]).mean(axis=0)
ensemble_auroc = roc_auc_score(y_val, ensemble_logratio)

# LogReg ensemble: concat PCA features from all layers, fit single LogReg
Z_train_all, Z_val_all = [], []
for li in CFG['layers']:
    scaler = results[li]['scaler']
    pca = results[li]['pca']
    Z_train_all.append(pca.transform(scaler.transform(pooled[li][idx_train])))
    Z_val_all.append(pca.transform(scaler.transform(pooled[li][idx_val])))
Z_train_cat = np.concatenate(Z_train_all, axis=1)  # [n, 3*64=192]
Z_val_cat = np.concatenate(Z_val_all, axis=1)
y_train = outcomes[idx_train]
logreg_concat = LogisticRegression(C=1.0, max_iter=3000, random_state=42).fit(Z_train_cat, y_train)
logreg_concat_scores = logreg_concat.decision_function(Z_val_cat)
logreg_concat_auroc = roc_auc_score(y_val, logreg_concat_scores)

print(f'{"=" * 60}')
print(f'SUMMARY — Validation AUROC (n={len(idx_val)})')
print(f'{"=" * 60}')
print(f'\n  Per-layer GMM density:')
for li in CFG['layers']:
    print(f'    L{li}:  {results[li]["auroc"]:.4f}')
best_layer_auroc = max(results[li]['auroc'] for li in CFG['layers'])
best_layer = max(CFG['layers'], key=lambda li: results[li]['auroc'])
print(f'    BEST: L{best_layer} = {best_layer_auroc:.4f}')
print(f'\n  Multi-layer ensemble (mean log-ratio):  {ensemble_auroc:.4f}   (Δ vs best = {ensemble_auroc - best_layer_auroc:+.4f})')
print(f'\n  LogReg baselines:')
for li in CFG['layers']:
    print(f'    L{li}:  {results[li]["logreg_auroc"]:.4f}')
print(f'    Concat all layers (192-d):  {logreg_concat_auroc:.4f}')
print(f'\n  GMM wins over LogReg per layer?')
for li in CFG['layers']:
    delta = results[li]['auroc'] - results[li]['logreg_auroc']
    print(f'    L{li}:  Δ = {delta:+.4f}  {"✅" if delta > 0.02 else ("=" if abs(delta) < 0.02 else "❌")}')


## Cell 7 — Plots

ROC curves per layer + ensemble, plus log-ratio distribution by outcome.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
ax = axes[0]
for li, color in zip(CFG['layers'], ['#3498db', '#9b59b6', '#2ecc71']):
    fpr, tpr, _ = roc_curve(y_val, results[li]['log_ratio'])
    ax.plot(fpr, tpr, color=color, label=f'L{li} GMM ({results[li]["auroc"]:.3f})')
# Ensemble
fpr_e, tpr_e, _ = roc_curve(y_val, ensemble_logratio)
ax.plot(fpr_e, tpr_e, color='#e74c3c', lw=2, label=f'Ensemble ({ensemble_auroc:.3f})')
# LogReg concat
fpr_l, tpr_l, _ = roc_curve(y_val, logreg_concat_scores)
ax.plot(fpr_l, tpr_l, color='#f39c12', ls='--', label=f'LogReg concat ({logreg_concat_auroc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC curves — per-layer GMM density + ensemble + LogReg baseline')
ax.legend(loc='lower right')

# Log-ratio distribution by outcome (best layer)
ax = axes[1]
best_lr = results[best_layer]['log_ratio']
for mask, label, color in [(y_val == 1, 'Correct', '#2ecc71'), (y_val == 0, 'Wrong', '#e74c3c')]:
    ax.hist(best_lr[mask], bins=30, alpha=0.6, label=f'{label} (n={mask.sum()})', color=color, density=True)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel(f'log p_C(z) - log p_W(z)  (layer L{best_layer})')
ax.set_ylabel('density')
ax.set_title(f'Log-ratio distribution by outcome — best layer L{best_layer}')
ax.legend()

plt.tight_layout()
plt.savefig('mcr_d_density_gmm.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n\u2705 Saved: mcr_d_density_gmm.png')

## Cell 8 — Verdict + save artifacts

In [ ]:
# Decision
ship = (best_layer_auroc > CFG['min_auroc_ship'] and
        ensemble_auroc >= best_layer_auroc + CFG['min_ensemble_gain'])

# Does GMM actually beat LogReg on best layer?
logreg_best = max(results[li]['logreg_auroc'] for li in CFG['layers'])
beats_linear = ensemble_auroc > logreg_concat_auroc + 0.01

print(f'{"=" * 60}')
if best_layer_auroc < 0.60:
    verdict = 'DENSITY_FAILS'
    msg = (f'Best layer AUROC {best_layer_auroc:.3f} < 0.60 — pooled residuals do not separate '
           f'correct from wrong well at response level. Consider token-level features or SAE encoding.')
    emoji = '\u274c'
elif not beats_linear:
    verdict = 'USE_LOGREG'
    msg = (f'LogReg concat ({logreg_concat_auroc:.3f}) ≥ GMM ensemble ({ensemble_auroc:.3f}). '
           f'Nonlinear density does not help. Use linear probe as R_density instead.')
    emoji = '\u26a0\ufe0f'
elif ship:
    verdict = 'SHIP_R_DENSITY'
    msg = (f'GMM ensemble AUROC {ensemble_auroc:.3f} beats best single layer by '
           f'{ensemble_auroc - best_layer_auroc:+.3f}. Multi-layer density ships.')
    emoji = '\u2705'
else:
    verdict = 'SHIP_SINGLE_LAYER'
    msg = (f'Best layer L{best_layer} AUROC {best_layer_auroc:.3f} is strong but ensemble adds '
           f'< {CFG["min_ensemble_gain"]}. Ship single-layer density; drop multi-layer term.')
    emoji = '\u2705'

print(f'{emoji}  VERDICT: {verdict}')
print(f'{"=" * 60}')
print(msg)

# Save artifacts
artifacts = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'hf_repo': CFG['hf_repo'],
    'layers': CFG['layers'],
    'pca_dim': CFG['pca_dim'],
    'gmm_k': CFG['gmm_k'],
    'gmm_cov': CFG['gmm_cov'],
    'n_train': int(len(idx_train)),
    'n_val': int(len(idx_val)),
    'per_layer_auroc': {f'L{li}': float(results[li]['auroc']) for li in CFG['layers']},
    'per_layer_logreg_auroc': {f'L{li}': float(results[li]['logreg_auroc']) for li in CFG['layers']},
    'per_layer_pca_var_kept': {f'L{li}': float(results[li]['var_kept']) for li in CFG['layers']},
    'best_layer': int(best_layer),
    'best_layer_auroc': float(best_layer_auroc),
    'ensemble_auroc': float(ensemble_auroc),
    'logreg_concat_auroc': float(logreg_concat_auroc),
    'gmm_beats_logreg': bool(beats_linear),
    'verdict': verdict,
    'message': msg,
}

with open('mcr_d_density_analysis.json', 'w') as f:
    json.dump(artifacts, f, indent=2)
print(f'\n\u2705 Saved: mcr_d_density_analysis.json')

# Pickle the fitted models for use in Stage G/H training
model_bundle = {}
for li in CFG['layers']:
    model_bundle[f'L{li}'] = {
        'scaler': results[li]['scaler'],
        'pca': results[li]['pca'],
        'gmm_c': results[li]['gmm_c'],
        'gmm_w': results[li]['gmm_w'],
        'logreg': results[li]['logreg'],
    }
model_bundle['logreg_concat'] = logreg_concat
with open('mcr_d_density_models.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)
print(f'\u2705 Saved: mcr_d_density_models.pkl (GMMs + PCAs + LogRegs)')

## Cell 9 — (optional) Per-discipline breakdown

Does density reward generalize across disciplines or is it domain-specific?

In [ ]:
print(f'{"=" * 60}')
print(f'Per-discipline AUROC (ensemble density)')
print(f'{"=" * 60}')
val_disciplines = disciplines[idx_val]
for disc in sorted(set(val_disciplines)):
    mask = val_disciplines == disc
    if mask.sum() < 10:
        continue
    y_d = y_val[mask]
    s_d = ensemble_logratio[mask]
    if len(set(y_d)) < 2:
        continue  # single class, no AUROC
    auroc_d = roc_auc_score(y_d, s_d)
    print(f'  {disc:28s}  n={mask.sum():3d}  acc={100*y_d.mean():.1f}%  AUROC={auroc_d:.3f}')